[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/fluids/projective_sphere_flow/projective_sphere_flow.ipynb)

# Projective Whitney Fields

This example computes projective Whitney fields, the corner hats and edge forms of a triangulated sphere as functions of the rays through its centre. A corner can move along its ray without moving its image on the sphere. That freedom can choose a different interpolation and dual mesh while the physical surface stays fixed. Steady heat diffusion on the sphere keeps the same heat sources and material as one corner moves, so that only the approximation changes.

A point of the sphere is modelled as a ray, any positive multiple of an antivector, and each lifted corner is one representative on its ray. A corner's hat is a quotient of two readings of a point by planes, `(reciprocals & points) / (section & points)`, whose reciprocal planes are joins of the other two corners. The hats need no metric, so the same construction serves the sphere in space and the plane in its degenerate projective algebra. The metric enters only with the physical surface: its quadrature measures and the material, a map from temperature gradients to currents.

In homogeneous coordinates, the hats are rational barycentric coordinates $\lambda_i(\mathbf x)=w_i\,b_i(\mathbf x)/\sum_j w_j\,b_j(\mathbf x)$, where the $b_i$ are the linear barycentric coordinates of the flat triangle, extended along its rays, and the weights $w_i=1/s_i$ are the inverse corner scales.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterable, Iterator

import numpy as np

from numga import NumpyContext, stack
from numga.algebras import VGA3D, PGA2D
from numga.sparse import spdiag
from examples import instantiate
from examples.util.animation import show_animation
from examples.fluids.projective_sphere_flow import render, scenarios

ga = VGA3D
ctx = NumpyContext(ga)
mv = ctx.multivector
mesh_module = instantiate("examples.util.mesh", ga)
Mesh, as_scalar = mesh_module.Mesh, mesh_module.as_scalar
Scalar = ga.gatype.scalar()                         # Scalar+
Plane = ga.gatype.vector()                          # Vector+
Point = ga.gatype.antivector()                      # Antivector-
Conductivity = ga.gatype((Point, Plane))            # Antivector- <- Vector+

In [ ]:
n_levels = scenarios.n_levels
corner_scales = scenarios.corner_scales
display, display_faces = scenarios.triangle_grid(scenarios.n_display_divisions)
mesh, vertices = Mesh.icosphere(n_levels)
render.draw_mesh(mesh, vertices);

## Fields live on rays

The three reciprocal planes read the lifted corners. Dividing by their sum makes their readings add to one everywhere on the projected triangle. The resulting field is constant along each ray; its derivative ignores radial motion.

In [ ]:
corners = vertices.batch()[mesh.faces[0]]                             # [corners] Point-
lifted = corners * corner_scales                                      # [corners] Point-
first, second, third = lifted                                         # [] Point-, Point-, Point-
volume = first & second & third                                       # [] Volume-
reciprocals = stack((second & third, third & first, first & second)) / volume   # [corners] Plane+
section = reciprocals.sum(axis=0)                                     # [] Plane+
points = (corners[None, :] * display).sum(axis=1).normalized()        # [samples] Point-
denominator = section & points                                        # [samples] Scalar+
hats = (reciprocals[:, None] & points) / denominator                  # [corners, samples] Scalar+
# This quotient derivative kills radial motion, before any metric is supplied.
gradients = (reciprocals[:, None] - hats * section) / denominator     # [corners, samples] Plane+
edge_planes = reciprocals[[1, 2, 0]] ^ reciprocals[[2, 0, 1]]         # [corners] Bivector+
edge_forms = (edge_planes[:, None] & points) / denominator.squared()  # [corners, samples] Plane+

The denominator matters: `reciprocals & points` alone changes along a ray and does not reproduce a constant field on the sphere. The edge reading is `edge_forms & displacement`, equivalently the alternating product of a hat and another hat's derivative. It depends on motion in the projective triangle, not on motion along the ray.

Changing the corner scales changes this rational interpolation. That is a choice of approximation space, not a deformation of the sphere. Positive scales keep the denominator away from zero inside each triangle; very unequal scales can still make a poor approximation.

## Same physical diffusion, different representation

Heat enters the northern hemisphere and leaves the southern hemisphere. With uniform conductivity on the unit sphere, the steady temperature is its height. The material response is integrated on that sphere even when the lifted triangles tilt. Only the trial fields change.

In [ ]:
n_frames = scenarios.n_frames
log_radius_amplitude = scenarios.log_radius_amplitude
frame_duration_ms = scenarios.frame_duration_ms
quadrature_vertices, quadrature_faces = scenarios.triangle_grid(scenarios.n_quadrature_divisions)
quadrature = quadrature_vertices[quadrature_faces].mean(axis=1)
chosen_vertex = int((mv.x & vertices.batch()).argmax())
phases = np.linspace(0, 2 * np.pi, n_frames, endpoint=False)
radii = np.exp(log_radius_amplitude * np.sin(phases[:, None]) * (np.arange(mesh.n_vertices) == chosen_vertex))

The adjugate carries the material response back through the same basis used for temperature, so constants produce no current and dissipation is nonnegative.

In [ ]:
def diffusion(mesh: Mesh, corners: Point, samples: Point, measures: Scalar,
              source: Scalar, conductivity: Conductivity) -> tuple[Scalar, Plane]:
    """The vertex potential solves steady diffusion under a prescribed source on a closed surface; the face planes interpolate it.

    Samples and positive quadrature measures belong to the physical surface; only the interpolation depends on the lifted corners.
    """
    first, second, third = corners                                   # Point-[faces], Point-[faces], Point-[faces]
    volume = first & second & third                                  # Volume-[faces]
    reciprocals = stack((second & third, third & first, first & second)) / volume   # [corners] Plane+[faces]
    section = reciprocals.sum(axis=0)                                # Plane+[faces]
    denominator = section & samples                                  # [samples] Scalar+[faces]
    hats = (reciprocals[None, :] & samples[:, None]) / denominator[:, None]   # [samples, corners] Scalar+[faces]
    # The derivative of the quotient annihilates the radial direction.
    gradients = (reciprocals[None, :] - hats * section) / denominator[:, None]   # [samples, corners] Plane+[faces]
    interpolation = (spdiag(hats * Scalar) * mesh.incidence[2, 0]).sum(axis=1)  # [samples] (Scalar+ <- Scalar+)[faces <- vertices]
    gradient = (spdiag(gradients * Scalar) * mesh.incidence[2, 0]).sum(axis=1)  # [samples] (Plane+ <- Scalar+)[faces <- vertices]

    # The gradient's adjugate pulls the physical material response back to the vertex potentials, integrated on the surface.
    response = spdiag(measures * conductivity)                       # [samples] (Point- <- Plane+)[faces <- faces]
    stiffness = gradient.adjugate()(response(gradient)).sum(axis=0)  # (Amount- <- Scalar+)[vertices <- vertices]
    load = interpolation.adjugate()((measures * source).dual()).sum(axis=0)   # Amount-[vertices]
    mass = interpolation.adjugate()(measures.dual()).sum(axis=0)     # Amount-[vertices]
    # A closed surface admits a solution only for balanced sources; remove the quadrature mean.
    load = load - mass * (load.dual_inverse().sites.sum() / mass.dual_inverse().sites.sum())
    pin = spdiag(as_scalar(np.arange(mesh.n_vertices) == 0).field() * Scalar.dual())   # (Amount- <- Scalar+)[vertices <- vertices]
    potential = (stiffness + pin).solve(load)                        # Scalar+[vertices]
    potential = potential - (potential & mass).sites.sum() / mass.dual_inverse().sites.sum()
    numerator = (reciprocals * (mesh.incidence[2, 0] * potential)).sum(axis=0)  # Plane+[faces]
    return potential, numerator

The face plane pairs to one with its lifted corners, and its pole `section.dual()` determines the projected dual mesh.

This cell takes about seven seconds, most of it drawing the animation.

In [ ]:
def spherical_frames(mesh: Mesh, vertices: Point, radii: Iterable[np.ndarray],
                     quadrature: np.ndarray, display: np.ndarray
                     ) -> Iterator[tuple[Point, Point, Point, Scalar]]:
    """Each frame solves the same source-and-sink problem again, under the interpolation its corner scales choose."""
    corners = mesh.incidence[2, 0] * vertices                              # [corners] Point-[faces]
    first, second, third = corners                                         # Point-[faces], Point-[faces], Point-[faces]
    volume = first & second & third                                        # Volume-[faces]
    quadrature_chords = (corners[None, :] * quadrature).sum(axis=1)        # [samples] Point-[faces]
    samples = quadrature_chords.normalized()                               # [samples] Point-[faces]
    # Radial projection expands chord area by the inverse cube of the distance.
    measures = volume.norm() / (2 * len(quadrature) * quadrature_chords.norm() ** 3)   # [samples] Scalar+[faces]
    source = 2 * (mv.z & samples)                                          # [samples] Scalar+[faces]
    conductivity = Plane.dual()                                            # [] Point- <- Plane+
    display_points = (corners[None, :] * display).sum(axis=1).normalized() # [samples] Point-[faces]
    for scales in radii:
        lifted = vertices * as_scalar(scales).field()                      # Point-[vertices]
        lifted_corners = mesh.incidence[2, 0] * lifted                     # [corners] Point-[faces]
        potential, numerator = diffusion(mesh, lifted_corners, samples, measures, source, conductivity)
        first, second, third = lifted_corners                              # Point-[faces], Point-[faces], Point-[faces]
        reciprocals = stack((second & third, third & first, first & second)) / (first & second & third)   # [corners] Plane+[faces]
        section = reciprocals.sum(axis=0)                                  # Plane+[faces]
        values = (numerator & display_points) / (section & display_points) # [samples] Scalar+[faces]
        yield lifted, section.dual().normalized(), display_points, values

frames = spherical_frames(mesh, vertices, radii, quadrature, display)
show_animation(render.animate_sphere(mesh, display_faces, frames), "projective_sphere_weights", frame_duration_ms)

The left panel shows the blue representatives and the orange polar dual projected onto the sphere; the right shows the computed temperature on the fixed sphere. The animation varies the representation while physical time stays fixed, and the temperature's departure from the height field is discretization error, not a change in the physical problem.

The inverse corner radii are the weights of a weighted dual mesh: changing them moves the dual while leaving the spherical vertices and great-circle edges fixed, and here the same weights also choose the rational interpolation. The material response comes from those rational fields integrated over the sphere, not from ratios of dual to primal edge lengths. Every admissible weighting represents the same physical problem, with its own approximation error. [Weighted triangulations](https://www.geometry.caltech.edu/pubs/dGMMD14.pdf) and [Hodge-optimized triangulations](https://pages.saclay.inria.fr/mathieu.desbrun/pubs/MMdGD11.pdf) use related freedom to improve primal/dual discretizations.

## The same quotient in x, y, w

In the degenerate algebra the physical surface is the finite affine slice. Multiplying all coordinates of a point by a positive number leaves that point fixed after normalization, while a direction has zero pairing with `mv.w`. Changing only `w` while holding `x` and `y` fixed moves the finite point; it is not a change of representative. The reciprocal planes and quotient derivatives need only joins and scalar division: no inverse of the degenerate metric occurs.

In [ ]:
ga = PGA2D
mv = NumpyContext(ga).multivector
Plane = ga.gatype.vector()                                             # Vector+
corner_values = scenarios.corner_values
planar_corners = scenarios.planar_corners                              # [corners] Point-

In [ ]:
planar_points = (planar_corners[None, :] * display).sum(axis=1)       # [samples] Point-
lifted = planar_corners * corner_scales                                # [corners] Point-
first, second, third = lifted                                          # [] Point-, Point-, Point-
reciprocals = stack((second & third, third & first, first & second)) / (first & second & third)   # [corners] Plane+
section = reciprocals.sum(axis=0)                                      # [] Plane+
planar_hats = (reciprocals[:, None] & planar_points) / (section & planar_points)   # [corners, samples] Scalar+
planar_gradients = (reciprocals[:, None] - planar_hats * section) / (section & planar_points)   # [corners, samples] Plane+
potential = (planar_hats * corner_values[:, None]).sum(axis=0)         # [samples] Scalar+
gradient = (planar_gradients * corner_values[:, None]).sum(axis=0)     # [samples] Plane+
# The material reads spatial directions, leaving w out of distance without inverting it.
directions = stack((mv.x.dual(), mv.y.dual()))                         # [axes] Point-
conductivity = (directions * (Plane & directions)).sum(axis=0)         # [] Point- <- Plane+
current = -conductivity(gradient)                                      # [samples] Point-
render.draw_planar(planar_points, display_faces, potential, current);

Colour shows the interpolated potential and the arrows the current in that physical plane. The material responds to spatial motion; the homogeneous coordinate is not an extra distance.

There are two uses of scale. Keeping `section = reciprocals.sum(axis=0)` lets the corner scales choose different rational hats, as in the spherical experiment. Keeping a physical chart fixed and compensating the reciprocal planes for the corner weights instead makes the rescaling purely a change of representative: every hat is unchanged.

Neither construction requires nearly equal weights. A zero denominator marks a chart boundary; points at infinity are valid projective points, but they are not finite locations of this planar diffusion problem.

In [ ]:
# The physical affine chart stays fixed when only representatives change.
chart = mv.w                                                              # [] Plane+
weighted_reciprocals = reciprocals * (chart & lifted)                     # [corners] Plane+
fixed_hats = (weighted_reciprocals[:, None] & planar_points) / (chart & planar_points)  # [corners, samples] Scalar+
# These are the ordinary affine hats, regardless of the independent corner scales.
first, second, third = planar_corners                                     # [] Point-, Point-, Point-
affine_reciprocals = stack((second & third, third & first, first & second)) / (first & second & third)   # [corners] Plane+
affine_hats = affine_reciprocals[:, None] & planar_points                 # [corners, samples] Scalar+

In [ ]:
# checks
np.testing.assert_allclose(hats.sum(axis=0).to_array(), 1, atol=1e-12)
np.testing.assert_allclose((gradients & points).to_array(), 0, atol=1e-12)
alternating = hats[[1, 2, 0]] * gradients[[2, 0, 1]] - hats[[2, 0, 1]] * gradients[[1, 2, 0]]
np.testing.assert_allclose((edge_forms - alternating).kernel, 0, atol=1e-12)
np.testing.assert_allclose((mv.w & current).to_array(), 0, atol=1e-12)
np.testing.assert_allclose((fixed_hats - affine_hats).to_array(), 0, atol=1e-12)